In [ ]:
%pip install pandas numpy matplotlib seaborn scikit-learn xgboost

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

from xgboost import XGBClassifier

# Load dataset
df = pd.read_csv("loan_data.csv")

# Clean column names
df.columns = df.columns.str.strip()

# Convert application date
df["application_date"] = pd.to_datetime(
    df["application_date"], errors="coerce"
)

# Extract year and month
df["application_year"] = df["application_date"].dt.year
df["application_month"] = df["application_date"].dt.month

# Drop original date
df.drop(columns=["application_date"], inplace=True)

# Separate features and target
X = df.drop(columns=["loan_status"])
y = df["loan_status"]

# Identify columns
numeric_features = [
    "age", "income", "loan_amount",
    "application_year", "application_month"
]

categorical_features = ["employment", "education"]

# Preprocessing
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# Split dataset
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Models
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=3, random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=4,
        bootstrap=True,
        oob_score=True,
        random_state=42
    )
}

results = []
trained_models = {}

# Train first three classifiers
for name, model in models.items():

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])

    pipe.fit(X_train, y_train)

    predictions = pipe.predict(X_val)

    score = accuracy_score(y_val, predictions)

    results.append({
        "Model": name,
        "Validation Score": score
    })

    trained_models[name] = pipe

    print("\n", name)
    print("Validation Accuracy:", score)

    if name == "Random Forest":
        print("OOB Score:", model.oob_score_)

# XGBoost with early stopping
# Fit preprocessing on training data only
preprocessor.fit(X_train)

X_val_transformed = preprocessor.transform(X_val)

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    eval_metric="logloss",
    early_stopping_rounds=10,
    random_state=42
)

xgb_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", xgb_model)
])

xgb_pipe.fit(
    X_train,
    y_train,
    classifier__eval_set=[
        (X_val_transformed, y_val)
    ],
    classifier__verbose=False
)

xgb_predictions = xgb_pipe.predict(X_val)

xgb_score = accuracy_score(y_val, xgb_predictions)

results.append({
    "Model": "XGBoost",
    "Validation Score": xgb_score
})

trained_models["XGBoost"] = xgb_pipe

print("\nXGBoost Validation Accuracy:", xgb_score)
print("Best Iteration:", xgb_model.best_iteration)

# Results table
results_df = pd.DataFrame(results)

print("\nCLASSIFICATION RESULTS")
print(results_df.to_string(index=False))


In [ ]:

# Get Logistic Regression pipeline
log_model = trained_models["Logistic Regression"]

# Select 5 validation rows
sample = X_val.iloc[:5]

# Probabilities
probabilities = log_model.predict_proba(sample)

# Predicted classes
predictions = log_model.predict(sample)

comparison = pd.DataFrame({
    "Actual": y_val.iloc[:5].values,
    "Probability_0": probabilities[:, 0],
    "Probability_1": probabilities[:, 1],
    "Predicted": predictions
})

print(comparison)


In [ ]:

rf_model = trained_models["Random Forest"]

feature_names = rf_model.named_steps[
    "preprocessor"
].get_feature_names_out()

rf_importance = rf_model.named_steps[
    "classifier"
].feature_importances_

rf_features = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf_importance
})

rf_features = rf_features.sort_values(
    by="Importance", ascending=False
).head(5)

print("TOP 5 RANDOM FOREST FEATURES")
print(rf_features)

sns.barplot(
    data=rf_features,
    x="Importance",
    y="Feature"
)
plt.title("Top 5 Random Forest Feature Importance")
plt.show()


In [ ]:

xgb_importance = xgb_pipe.named_steps[
    "classifier"
].feature_importances_

xgb_features = pd.DataFrame({
    "Feature": feature_names,
    "Importance": xgb_importance
})

xgb_features = xgb_features.sort_values(
    by="Importance", ascending=False
).head(5)

print("TOP 5 XGBOOST FEATURES")
print(xgb_features)

sns.barplot(
    data=xgb_features,
    x="Importance",
    y="Feature"
)
plt.title("Top 5 XGBoost Feature Importance")
plt.show()
